# Comparing Models Performance

This notebook reads `results_final_v2.csv` and generates comparison plots for the 4 models:

- Decision Tree
- Random Forest
- GBT
- MLP (PyTorch)

The plot colors and numeric annotations are styled to match the visual style used in `04_modeling.ipynb`.


In [ ]:
import pandas as pd                            
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 10

# Try common locations automatically
candidate_paths = [
    Path("results_final_v2.csv"),
    Path("../data/results_final_v2.csv"),
    Path("../results_final_v2.csv"),
    Path("/mnt/data/results_final_v2.csv"),
]

csv_path = None
for p in candidate_paths:
    if p.exists():
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError(
        "Could not find results_final_v2.csv. Put the CSV in the same folder as this notebook "
        "or update candidate_paths manually."
    )

df = pd.read_csv(csv_path)
df

In [ ]:
# Basic checks
required_cols = ["Experiment", "Model", "AUC", "AUC_std", "Acc", "F1"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

plot_df = df.copy()

preferred_model_order = ["Decision Tree", "Random Forest", "GBT", "MLP", "MLP (PyTorch)"]
model_order = [m for m in preferred_model_order if m in plot_df["Model"].unique()]
extra_models = [m for m in plot_df["Model"].unique() if m not in model_order]
model_order += extra_models

preferred_exp_order = ["Sim", "Real"]
exp_order = [e for e in preferred_exp_order if e in plot_df["Experiment"].unique()]
extra_exps = [e for e in plot_df["Experiment"].unique() if e not in exp_order]
exp_order += extra_exps

plot_df["Model"] = pd.Categorical(plot_df["Model"], categories=model_order, ordered=True)
plot_df["Experiment"] = pd.Categorical(plot_df["Experiment"], categories=exp_order, ordered=True)
plot_df = plot_df.sort_values(["Experiment", "Model"]).reset_index(drop=True)

# Color style matched to 04_modeling.ipynb
SIM_PALETTE = ["#aed6f1", "#5dade2", "#2e86c1", "#1b4f72"]
REAL_PALETTE = ["#a9dfbf", "#58d68d", "#28b463", "#145a32"]
METRIC_PALETTE = {"AUC": "#5dade2", "Acc": "#58d68d", "F1": "#f5b041"}

def get_palette(experiment, n):
    palette = REAL_PALETTE if str(experiment).lower() == "real" else SIM_PALETTE
    if n <= len(palette):
        return palette[:n]
    return [palette[i % len(palette)] for i in range(n)]

def annotate_bars(ax, bars, values=None, fmt="{:.3f}", offset=0.008, fontsize=10):
    """Add bold numeric labels on top of bars, matching 04_modeling.ipynb style."""
    if values is None:
        values = [bar.get_height() for bar in bars]
    for bar, v in zip(bars, values):
        if pd.isna(v):
            continue
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            fmt.format(float(v)),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            fontweight="bold",
        )

output_dir = Path("../data/model_comparison_plots")
output_dir.mkdir(parents=True, exist_ok=True)

plot_df


In [ ]:
def grouped_bar(metric, title, ylabel, error_metric=None, figsize=(11, 6)):
    fig, axes = plt.subplots(1, len(exp_order), figsize=figsize, sharey=True)
    if len(exp_order) == 1:
        axes = [axes]

    for ax, exp in zip(axes, exp_order):
        sub = (
            plot_df[plot_df["Experiment"] == exp]
            .sort_values(metric, ascending=False)
            .reset_index(drop=True)
        )
        colors = get_palette(exp, len(sub))
        bars = ax.bar(sub["Model"], sub[metric], color=colors, width=0.5)

        if error_metric is not None and error_metric in sub.columns:
            errs = sub[error_metric].fillna(0).values
            ax.errorbar(
                range(len(sub)),
                sub[metric].values,
                yerr=errs,
                fmt="none",
                color="black",
                capsize=5,
                linewidth=2,
            )

        if metric == "AUC":
            ax.axhline(0.5, color="red", linestyle="--", linewidth=1, alpha=0.5, label="Random (AUC=0.5)")
            ax.set_ylim(0.4, 1.05)
            ax.legend(fontsize=9)
        else:
            ymax = min(1.05, max(1.0, sub[metric].max() + 0.08))
            ymin = max(0.0, min(sub[metric].min() - 0.08, 0.0))
            ax.set_ylim(ymin, ymax)

        ax.set_title(f"{exp} Experiment ({ylabel})", fontsize=12)
        ax.set_ylabel(ylabel)
        annotate_bars(ax, bars, sub[metric].values)
        ax.tick_params(axis="x", rotation=0)

    plt.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


def metric_by_experiment(experiment, metrics=("AUC", "Acc", "F1"), figsize=(11, 6)):
    sub = (
        plot_df[plot_df["Experiment"] == experiment]
        .set_index("Model")
        .reindex(model_order)
        .dropna(subset=list(metrics), how="all")
    )

    fig, axes = plt.subplots(1, len(metrics), figsize=figsize, sharey=False)
    if len(metrics) == 1:
        axes = [axes]

    exp_colors = get_palette(experiment, len(sub))

    for ax, metric in zip(axes, metrics):
        ordered = sub[[metric]].sort_values(metric, ascending=False).dropna()
        colors = exp_colors[:len(ordered)]
        bars = ax.bar(ordered.index, ordered[metric].values, color=colors, width=0.5)

        if metric == "AUC" and "AUC_std" in sub.columns:
            std_sub = sub.loc[ordered.index, "AUC_std"].fillna(0).values
            ax.errorbar(
                range(len(ordered)),
                ordered[metric].values,
                yerr=std_sub,
                fmt="none",
                color="black",
                capsize=5,
                linewidth=2,
            )
            ax.axhline(0.5, color="red", linestyle="--", linewidth=1, alpha=0.5, label="Random (AUC=0.5)")
            ax.set_ylim(0.4, 1.05)
            ax.legend(fontsize=8)
        else:
            ymax = min(1.05, max(1.0, ordered[metric].max() + 0.08))
            ymin = max(0.0, min(ordered[metric].min() - 0.08, 0.0))
            ax.set_ylim(ymin, ymax)

        ax.set_title(metric, fontsize=12)
        ax.set_ylabel(metric)
        annotate_bars(ax, bars, ordered[metric].values)
        ax.tick_params(axis="x", rotation=20)

    plt.suptitle(f"{experiment}: Model Comparison Across Metrics", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


def heatmap_table(metrics=("AUC", "AUC_std", "Acc", "F1"), figsize=(10, 4)):
    pivot = (
        plot_df.assign(Row=plot_df["Experiment"].astype(str) + " | " + plot_df["Model"].astype(str))
               .set_index("Row")[list(metrics)]
    )

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(pivot.values, aspect="auto")

    ax.set_xticks(np.arange(len(metrics)))
    ax.set_xticklabels(metrics)
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title("Performance Summary Heatmap")

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(
                j, i, f"{pivot.iloc[i, j]:.3f}",
                ha="center", va="center", fontweight="bold"
            )

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

    return pivot

## Plot 1–3: Separate model comparison for Sim and Real

In [ ]:

def plot_metric_two_panels(metric, ylabel, filename, error_metric=None, baseline=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes = np.atleast_1d(axes)

    for ax, exp_name in zip(axes, exp_order[:2]):
        sub = plot_df[plot_df["Experiment"] == exp_name].copy().sort_values(metric, ascending=False)
        colors = get_palette(exp_name, len(sub))

        yerr = sub[error_metric].values if error_metric is not None and error_metric in sub.columns else None
        bars = ax.bar(sub["Model"], sub[metric], color=colors, width=0.5)
        if yerr is not None:
            ax.errorbar(
                range(len(sub)),
                sub[metric].values,
                yerr=yerr,
                fmt="none",
                color="black",
                capsize=5,
                linewidth=2,
            )

        if metric == "AUC":
            ax.set_ylim(0.4, 1.05)
            if baseline is not None:
                ax.axhline(
                    baseline, color="red", linestyle="--", linewidth=1, alpha=0.5,
                    label=f"Random (AUC={baseline})"
                )
                ax.legend(fontsize=9)
        else:
            ymax = min(1.05, max(1.0, float(np.nanmax(sub[metric])) + 0.08))
            ymin = max(0.0, min(float(np.nanmin(sub[metric])) - 0.08, 0.0))
            ax.set_ylim(ymin, ymax)

        ax.set_title(f"{exp_name} Experiment ({metric})", fontsize=12)
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", linestyle="--", alpha=0.25)

        for tick in ax.get_xticklabels():
            tick.set_rotation(12)

        annotate_bars(ax, bars, values=sub[metric].values, fmt="{:.3f}", offset=0.01, fontsize=10)

    plt.suptitle(f"Model Comparison by {metric}", fontsize=13, y=1.02)
    plt.tight_layout()
    save_path = output_dir / filename
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

plot_metric_two_panels(metric="AUC", ylabel="AUC-ROC", filename="auc_two_panel.png", error_metric="AUC_std", baseline=0.5)
plot_metric_two_panels(metric="Acc", ylabel="Accuracy", filename="accuracy_two_panel.png")
plot_metric_two_panels(metric="F1", ylabel="F1 Score", filename="f1_two_panel.png")


## Plot 4: AUC grouped comparison across Sim and Real

In [ ]:

def grouped_auc_comparison():
    fig, ax = plt.subplots(figsize=(10, 6))

    x = np.arange(len(exp_order))
    width = 0.18 if len(model_order) >= 4 else 0.25

    grouped_bars = []
    for i, model in enumerate(model_order):
        sub = plot_df[plot_df["Model"] == model].set_index("Experiment").reindex(exp_order)
        vals = sub["AUC"].values
        errs = sub["AUC_std"].fillna(0).values
        xpos = x + (i - (len(model_order)-1)/2) * width
        color = SIM_PALETTE[i % len(SIM_PALETTE)]
        bars = ax.bar(xpos, vals, width, label=model, color=color)
        ax.errorbar(xpos, vals, yerr=errs, fmt="none", color="black", capsize=5, linewidth=2)
        grouped_bars.append((bars, vals))

    ax.set_xticks(x)
    ax.set_xticklabels(exp_order)
    ax.set_title("AUC Comparison Across Models", fontsize=12)
    ax.set_xlabel("Experiment")
    ax.set_ylabel("AUC-ROC")
    ax.set_ylim(0.4, 1.05)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=1, alpha=0.5, label="Random (AUC=0.5)")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.25)

    for bars, vals in grouped_bars:
        annotate_bars(ax, bars, values=vals, fmt="{:.3f}", offset=0.01, fontsize=9)

    plt.tight_layout()
    save_path = output_dir / "auc_grouped_comparison.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

grouped_auc_comparison()


## Plot 5: Metric comparison inside each experiment

In [ ]:

def metric_by_experiment(experiment, metrics=("AUC", "Acc", "F1"), figsize=(11, 6)):
    sub = plot_df[plot_df["Experiment"] == experiment].set_index("Model").reindex(model_order)

    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(model_order))
    width = 0.22

    all_bars = []
    for i, metric in enumerate(metrics):
        vals = sub[metric].values
        bars = ax.bar(
            x + (i - (len(metrics)-1)/2) * width,
            vals,
            width,
            label=metric,
            color=METRIC_PALETTE[metric]
        )
        all_bars.append((bars, vals))

    ax.set_xticks(x)
    ax.set_xticklabels(model_order, rotation=12)
    ax.set_title(f"{experiment}: Model Comparison Across Metrics", fontsize=12)
    ax.set_xlabel("Model")
    ax.set_ylabel("Score")
    ax.set_ylim(0.0, 1.05)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.25)

    for bars, vals in all_bars:
        annotate_bars(ax, bars, values=vals, fmt="{:.3f}", offset=0.01, fontsize=9)

    plt.tight_layout()
    save_path = output_dir / f"{str(experiment).lower()}_metrics_grouped.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

for exp in exp_order:
    metric_by_experiment(exp)


## Plot 6: Summary heatmap

In [ ]:

def heatmap_table(metrics=("AUC", "AUC_std", "Acc", "F1"), figsize=(10, 4)):
    pivot = (
        plot_df.assign(Row=plot_df["Experiment"].astype(str) + " | " + plot_df["Model"].astype(str))
               .set_index("Row")[list(metrics)]
    )

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(pivot.values, aspect="auto")

    ax.set_xticks(np.arange(len(metrics)))
    ax.set_xticklabels(metrics)
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title("Performance Summary Heatmap", fontsize=12)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.iloc[i, j]:.3f}", ha="center", va="center", fontsize=9, fontweight='bold')

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    save_path = output_dir / "performance_summary_heatmap.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

    return pivot

summary_table = heatmap_table()
summary_table
